# GardnAIze — Adaptation climatique des jardins V2
26/11/2026 · GDAI-PLANT

50 scénarios, dont 24 océaniques à adapter (10 transition, 14 semi-continentaux). Les profils modifiés sont synthétiques, non des observations de jardins réels. Aucun label de compatibilité ne doit être réutilisé sans réévaluation.

In [2]:
from pathlib import Path
import sys

print("Working directory :", Path.cwd())
print("Python executable :", sys.executable)

print("Source directe :", Path(
    "data/scenarios/garden-scenarios-final-v2.json"
).exists())

print("Source via gdai-plant :", Path(
    "gdai-plant/data/scenarios/garden-scenarios-final-v2.json"
).exists())

Working directory : /home/bsui/projects/GardnAIze/ai-service/plant-ingestion
Python executable : /home/bsui/projects/GardnAIze/gdai-plant/.venv/bin/python
Source directe : False
Source via gdai-plant : False


In [3]:
from pathlib import Path
import os

cwd = Path.cwd().resolve()

PROJECT_ROOT = next(
    (
        p for p in [cwd, *cwd.parents]
        if (p / "gdai-plant" / "data" / "scenarios"
            / "garden-scenarios-final-v2.json").is_file()
    ),
    None
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Impossible de localiser le projet GardnAIze"
    )

PLANT_ROOT = PROJECT_ROOT / "gdai-plant"

os.chdir(PLANT_ROOT)

print("Project root :", PROJECT_ROOT)
print("Plant root   :", PLANT_ROOT)
print("Working dir  :", Path.cwd())


import json, copy, hashlib, unicodedata, re
from pathlib import Path
from collections import Counter
SOURCE = Path('data/scenarios/garden-scenarios-final-v2.json')
OUTPUT = SOURCE.with_name('garden-scenarios-climate-v1.json')
REPORT = SOURCE.with_name('climate-adaptation-report-v1.json')
raw = SOURCE.read_bytes()
original = json.loads(raw)
print(type(original), list(original)[:10] if isinstance(original, dict) else len(original))
print(json.dumps(original[0] if isinstance(original,list) else original, ensure_ascii=False, indent=2)[:6000])

Project root : /home/bsui/projects/GardnAIze
Plant root   : /home/bsui/projects/GardnAIze/gdai-plant
Working dir  : /home/bsui/projects/GardnAIze/gdai-plant
<class 'dict'> ['schema_version', 'generator', 'source_files', 'summary', 'selection_policy', 'excluded', 'reviewed_near_duplicate_pairs', 'scenarios']
{
  "schema_version": "2.0",
  "generator": "OFFLINE_FINAL_SELECTION",
  "source_files": [
    "garden-scenarios-consolidated-v1.json",
    "pilot-24-targeted-v1.2.json"
  ],
  "summary": {
    "raw": 53,
    "selected": 50,
    "excluded": 3,
    "validation_accepted": 50
  },
  "selection_policy": "Deterministic MVP selection; synthetic hypotheses, not expert-validated observations or garden-plant compatibility labels.",
  "excluded": [
    {
      "id": "FR-TEM-001",
      "reason": "Near-duplicate of FR-PAR-01, excluded in v1."
    },
    {
      "id": "FR-TEM-019",
      "reason": "RAIN_GARDEN label for a natural wet depression remains unresolved; hold outside MVP."
    },
    

## 1. Configurer le schéma réel
Adapter les chemins ci-dessous après inspection. Les valeurs climatiques doivent correspondre aux enums du moteur. Les valeurs de température, humidité et eau doivent être définies et validées pour chaque scénario, sans inventer de mesures réelles.

In [ ]:
ROOT = "scenarios"  # None si liste, sinon clé contenant les scénarios
ID = 'id'
CLIMATE = 'climate_zone'  # chemins imbriqués possibles: context.climate_zone
TEMP = 'minimum_winter_temperature_c'
HUMIDITY = 'soil_moisture_seasonality'
WATER = 'water_availability'
TRANSITION = "TEMPERATE_TRANSITIONAL"
CONTINENTAL = "TEMPERATE_SEMI_CONTINENTAL"
def get(obj, path):
    for k in path.split('.'): obj = obj[k]
    return obj
def put(obj, path, value):
    keys = path.split('.')
    for k in keys[:-1]: obj = obj[k]
    assert keys[-1] in obj, path
    obj[keys[-1]] = value
def norm(x):
    return re.sub(r'[^a-z0-9]+','_',unicodedata.normalize('NFKD',str(x).lower()).encode('ascii','ignore').decode()).strip('_')
OCEANIC = {'oceanic','oceanique','oceanic_climate','oceanique_altere','oceanic_altered'}
FORBIDDEN = OCEANIC | {'mediterranean','mediterraneen','mountain','montagnard','alpine'}

import json

document = json.loads(raw)

rows = document["scenarios"]

assert isinstance(rows, list), "Les scénarios doivent être une liste"
assert len(rows) == 50, f"Nombre de jardins inattendu : {len(rows)}"

print("Type de rows :", type(rows))
print("Nombre de jardins :", len(rows))
print("Champs disponibles :", list(rows[0].keys()))

assert isinstance(rows,list) and len(rows)==50

ids = [r["id"] for r in rows]

assert len(ids) == 50
assert len(set(ids)) == 50, "Identifiants en double"

print("Nombre d'identifiants :", len(ids))
print("Identifiants uniques :", len(set(ids)))
print("Premiers identifiants :", ids[:5])

assert len(set(ids))==50
print(Counter(get(r,CLIMATE) for r in rows))

targets = [
    r for r in rows
    if r["climate_zone"] == "TEMPERATE_OCEANIC"
]

print("Nombre de jardins à adapter :", len(targets))

assert len(targets) == 24, (
    f"24 jardins attendus, {len(targets)} trouvés"
)

print("Identifiants des jardins à adapter :")

for r in targets:
    print(r["id"], r["climate_zone"])

assert len(targets)==24, 'Vérifier les enums et la version du dataset'
for r in targets: print(get(r,ID), get(r,CLIMATE), get(r,TEMP), get(r,HUMIDITY), get(r,WATER))

from collections import Counter

distribution = Counter(
    row["climate_zone"] for row in rows
)

for climate, count in distribution.items():
    print(f"{climate}: {count}")

assert distribution["TEMPERATE_OCEANIC"] == 24

print("\nValidation du schéma : OK")

Type de rows : <class 'list'>
Nombre de jardins : 50
Champs disponibles : ['id', 'sun_exposure', 'soil_drainage', 'water_availability', 'minimum_winter_temperature_c', 'maximum_height_cm', 'maximum_spread_cm', 'evergreen_preferred', 'origin', 'plausibility_reason', 'climate_zone', 'soil_texture', 'soil_moisture_seasonality', 'planting_context', 'terrain_type', 'enrichment_provenance']
Nombre d'identifiants : 50
Identifiants uniques : 50
Premiers identifiants : ['FR-PAR-01', 'FR-BRE-02', 'FR-AQU-03', 'FR-CTR-04', 'FR-EST-05']
Counter({'TEMPERATE_OCEANIC': 24, 'TEMPERATE_TRANSITIONAL': 15, 'TEMPERATE_SEMI_CONTINENTAL': 11})
Nombre de jardins à adapter : 24
Identifiants des jardins à adapter :
FR-BRE-02 TEMPERATE_OCEANIC
FR-AQU-03 TEMPERATE_OCEANIC
FR-NOR-06 TEMPERATE_OCEANIC
FR-TEM-002 TEMPERATE_OCEANIC
FR-TEM-003 TEMPERATE_OCEANIC
FR-TEM-004 TEMPERATE_OCEANIC
FR-TEM-005 TEMPERATE_OCEANIC
FR-TEM-008 TEMPERATE_OCEANIC
FR-TEM-009 TEMPERATE_OCEANIC
FR-TEM-012 TEMPERATE_OCEANIC
FR-TEM-013 TE

## 2. Préparer un plan éditable
La répartition initiale par identifiant est déterministe, mais arbitraire : modifier les affectations selon la cohérence géographique. Compléter les trois valeurs et la justification pour chaque jardin. Le plan est conservé entre exécutions.

In [28]:
import json
import pandas as pd

PLAN = SOURCE.with_name(
    "garden-climate-adaptation-plan-v1.json"
)

# 1. Créer le plan uniquement s'il n'existe pas
if not PLAN.exists():
    plan = [
        {
            "id": get(r, ID),
            "climate": TRANSITION if i < 10 else CONTINENTAL,
            "temperature": None,
            "humidity": None,
            "water": None,
            "reason": "",
            "reviewed": False
        }
        for i, r in enumerate(targets)
    ]

    PLAN.write_text(
        json.dumps(plan, ensure_ascii=False, indent=2),
        encoding="utf-8"
    )

    print("Nouveau plan créé.")
else:
    print("Plan existant conservé.")

# 2. Charger le contenu du fichier JSON
plan = json.loads(
    PLAN.read_text(encoding="utf-8")
)

# 3. Vérifier le contenu
assert isinstance(plan, list)
assert len(plan) == 24
assert len({p["id"] for p in plan}) == 24
assert {p["id"] for p in plan} == {
    get(r, ID) for r in targets
}

# 4. Construire le DataFrame à partir des données
df_plan = pd.DataFrame(plan)

# 5. Afficher le plan complet
display(df_plan)

# 6. Afficher les statistiques
print("Nombre de modifications :", len(df_plan))

print(
    "Nombre de modifications validées :",
    df_plan["reviewed"].eq(True).sum()
)

print("\nDistribution climatique cible :")
print(df_plan["climate"].value_counts())

print("\nFichier du plan :", PLAN.resolve())

print(
    "\nModifier et valider ce fichier avant "
    "d'exécuter les cellules suivantes."
)

Plan existant conservé.


,id,climate,temperature,humidity,water,reason,reviewed,region,plausibility_reason
0,FR-BRE-02,TEMPERATE_TRANSITIONAL,-8,SEASONAL_WET,HIGH,Hypothèse de relocalisation synthétique en Île...,True,Île-de-France,Synthetic relocated scenario in lowland Île-de...
1,FR-AQU-03,TEMPERATE_TRANSITIONAL,-8,DRY_SUMMER,LOW,Hypothèse de relocalisation synthétique en Cen...,True,Centre-Val de Loire,Synthetic relocated scenario in lowland Centre...
2,FR-NOR-06,TEMPERATE_TRANSITIONAL,-9,WELL_DRAINED,HIGH,Hypothèse de relocalisation synthétique en Île...,True,Île-de-France,Synthetic relocated scenario in lowland Île-de...
3,FR-TEM-002,TEMPERATE_TRANSITIONAL,-8,WELL_DRAINED,MEDIUM,Hypothèse de relocalisation synthétique en Cen...,True,Centre-Val de Loire,Synthetic relocated scenario in lowland Centre...
4,FR-TEM-003,TEMPERATE_TRANSITIONAL,-9,CONSISTENTLY_MOIST,MEDIUM,Hypothèse de relocalisation synthétique en Île...,True,Île-de-France,Synthetic relocated scenario in lowland Île-de...
5,FR-TEM-004,TEMPERATE_TRANSITIONAL,-8,WELL_DRAINED,HIGH,Hypothèse de relocalisation synthétique en Cen...,True,Centre-Val de Loire,Synthetic relocated scenario in lowland Centre...
6,FR-TEM-005,TEMPERATE_TRANSITIONAL,-9,SEASONAL_WET,MEDIUM,Hypothèse de relocalisation synthétique en Hau...,True,Hauts-de-France,Synthetic relocated scenario in lowland Hauts-...
7,FR-TEM-008,TEMPERATE_TRANSITIONAL,-9,DRY_SUMMER,MEDIUM,Hypothèse de relocalisation synthétique en Cen...,True,Centre-Val de Loire,Synthetic relocated scenario in lowland Centre...
8,FR-TEM-009,TEMPERATE_TRANSITIONAL,-8,CONSISTENTLY_MOIST,MEDIUM,Hypothèse de relocalisation synthétique en Île...,True,Île-de-France,Synthetic relocated scenario in lowland Île-de...
9,FR-TEM-012,TEMPERATE_TRANSITIONAL,-9,CONSISTENTLY_MOIST,MEDIUM,Hypothèse de relocalisation synthétique en Cen...,True,Centre-Val de Loire,Synthetic relocated scenario in lowland Centre...


Nombre de modifications : 24
Nombre de modifications validées : 24

Distribution climatique cible :
climate
TEMPERATE_SEMI_CONTINENTAL    14
TEMPERATE_TRANSITIONAL        10
Name: count, dtype: int64

Fichier du plan : /home/bsui/projects/GardnAIze/gdai-plant/data/scenarios/garden-climate-adaptation-plan-v1.json

Modifier et valider ce fichier avant d'exécuter les cellules suivantes.


In [29]:
plan = json.loads(
    PLAN.read_text(encoding="utf-8")
)

assert len(plan) == 24
assert len({p["id"] for p in plan}) == 24

for p in plan:
    assert p["reason"].strip()
    assert p["temperature"] is not None
    assert p["humidity"] is not None
    assert p["water"] is not None

    assert p["climate"] in {
        "TEMPERATE_TRANSITIONAL",
        "TEMPERATE_SEMI_CONTINENTAL"
    }

# Validation humaine du plan approuvé
for p in plan:
    p["reviewed"] = True

PLAN.write_text(
    json.dumps(plan, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

print("Modifications validées :", len(plan))

Modifications validées : 24


In [30]:
import pandas as pd

df_targets = pd.DataFrame(targets)

columns = [
    "id",
    "climate_zone",
    "minimum_winter_temperature_c",
    "soil_moisture_seasonality",
    "water_availability",
    "soil_drainage",
    "soil_texture",
    "plausibility_reason"
]

with pd.option_context(
    "display.max_rows", 50,
    "display.max_columns", None,
    "display.max_colwidth", None
):
    display(df_targets[columns])

REVIEW_FILE = SOURCE.with_name(
    "garden-climate-review-input-v1.json"
)

REVIEW_FILE.write_text(
    json.dumps(targets, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

print("Fichier généré :", REVIEW_FILE.resolve())
print("Nombre de jardins :", len(targets))



,id,climate_zone,minimum_winter_temperature_c,soil_moisture_seasonality,water_availability,soil_drainage,soil_texture,plausibility_reason
0,FR-BRE-02,TEMPERATE_OCEANIC,-4,SEASONAL_WET,HIGH,MODERATE,MIXED,Residential boundary bed along a wooden fence in maritime Brittany with frequent ocean-influenced precipitation and mild winters.
1,FR-AQU-03,TEMPERATE_OCEANIC,-5,DRY_SUMMER,LOW,GOOD,MIXED,Sandy loam border flanking a sun-drenched paved patio in southwestern France where summer precipitation is minimal.
2,FR-NOR-06,TEMPERATE_OCEANIC,-6,WELL_DRAINED,HIGH,GOOD,MIXED,Gently sloped residential side yard in Normandy benefiting from regular Atlantic rain systems and natural gravitational drainage.
3,FR-TEM-002,TEMPERATE_OCEANIC,-4,WELL_DRAINED,MEDIUM,GOOD,SAND,Coastal residential garden border sheltered by an existing fence under mild oceanic conditions with regular precipitation.
4,FR-TEM-003,TEMPERATE_OCEANIC,-8,CONSISTENTLY_MOIST,MEDIUM,MODERATE,SILT,Boundary property line hedge planting in a Normandy housing estate with fertile silty plains soil.
5,FR-TEM-004,TEMPERATE_OCEANIC,-6,WELL_DRAINED,HIGH,GOOD,MIXED,Constructed timber raised bed with supplemented topsoil in a Loire Valley family garden receiving municipal irrigation.
6,FR-TEM-005,TEMPERATE_OCEANIC,-9,SEASONAL_WET,MEDIUM,MODERATE,SILT,Suburban backyard edge partially shaded by neighboring deciduous canopy in the northern chalk/silt plain.
7,FR-TEM-008,TEMPERATE_OCEANIC,-6,DRY_SUMMER,MEDIUM,MODERATE,CLAY,"Formal hedge boundary on heavy lowland clay plain south of the Loire, susceptible to dry spells."
8,FR-TEM-009,TEMPERATE_OCEANIC,-5,CONSISTENTLY_MOIST,MEDIUM,MODERATE,LOAM,West-facing house foundation bed in lower Normandy with regular Atlantic airflows and consistent soil humidity.
9,FR-TEM-012,TEMPERATE_OCEANIC,-4,CONSISTENTLY_MOIST,MEDIUM,GOOD,LOAM,"Inland Breton garden border bordering a mature oak copse, featuring acid-leaning silty loam."


Fichier généré : /home/bsui/projects/GardnAIze/gdai-plant/data/scenarios/garden-climate-review-input-v1.json
Nombre de jardins : 24


## 3. Appliquer les modifications après revue
Contrôler aussi ville/région, description, photo éventuelle et cohérence du sol. Si une localisation réelle contredit le nouveau climat, ne pas valider le scénario avant correction explicite. Les champs hors climat restent inchangés.

In [31]:
plan = json.loads(PLAN.read_text(encoding='utf-8'))
assert len(plan)==24 and {p['id'] for p in plan}=={get(r,ID) for r in targets}

print("TRANSITION :", repr(TRANSITION))
print("CONTINENTAL :", repr(CONTINENTAL))

print("Distribution réelle :")
print(Counter(p["climate"] for p in plan))

print("Distribution attendue :")
print(Counter({TRANSITION: 10, CONTINENTAL: 14}))

assert Counter(p['climate'] for p in plan)==Counter({TRANSITION:10,CONTINENTAL:14})
for p in plan:
    assert p['reviewed'] is True and p['reason'].strip(), p['id']
    assert all(p[k] is not None for k in ('temperature','humidity','water')), p['id']
    assert isinstance(p['temperature'],(int,float)) and not isinstance(p['temperature'],bool), p['id']
updated = copy.deepcopy(original)
new_rows = updated if ROOT is None else get(updated,ROOT)
lookup = {p['id']:p for p in plan}
changes=[]
for r in new_rows:
    ident=get(r,ID)
    if ident not in lookup: continue
    p=lookup[ident]
    before=copy.deepcopy(r)
    for field,key in [(CLIMATE,'climate'),(TEMP,'temperature'),(HUMIDITY,'humidity'),(WATER,'water')]:
        put(r,field,p[key])
    changes.append({'id':ident,'before':{f:get(before,f) for f in (CLIMATE,TEMP,HUMIDITY,WATER)},'after':{f:get(r,f) for f in (CLIMATE,TEMP,HUMIDITY,WATER)},'reason':p['reason']})
assert len(new_rows)==50 and [get(r,ID) for r in new_rows]==ids
assert sum(a!=b for a,b in zip(rows,new_rows))==24
assert not any(norm(get(r,CLIMATE)) in FORBIDDEN for r in new_rows)
# Détecter les doublons de contenu indépendamment de l'identifiant.
def fingerprint(r):
    c=copy.deepcopy(r); put(c,ID,None)
    return json.dumps(c,sort_keys=True,ensure_ascii=False)
assert len(set(map(fingerprint,new_rows)))==50
print('Avant:',Counter(get(r,CLIMATE) for r in rows))
print('Après:',Counter(get(r,CLIMATE) for r in new_rows))

TRANSITION : 'TEMPERATE_TRANSITIONAL'
CONTINENTAL : 'TEMPERATE_SEMI_CONTINENTAL'
Distribution réelle :
Counter({'TEMPERATE_SEMI_CONTINENTAL': 14, 'TEMPERATE_TRANSITIONAL': 10})
Distribution attendue :
Counter({'TEMPERATE_SEMI_CONTINENTAL': 14, 'TEMPERATE_TRANSITIONAL': 10})


TypeError: string indices must be integers, not 'str'

In [ ]:
assert SOURCE.read_bytes()==raw
assert not OUTPUT.exists() and not REPORT.exists(), 'Exports existants : choisir une nouvelle version'
encoded=json.dumps(updated,ensure_ascii=False,indent=2).encode('utf-8')
report={'source':str(SOURCE),'source_sha256':hashlib.sha256(raw).hexdigest(),'output_sha256':hashlib.sha256(encoded).hexdigest(),'before':dict(Counter(get(r,CLIMATE) for r in rows)),'after':dict(Counter(get(r,CLIMATE) for r in new_rows)),'modified':24,'unchanged':26,'changes':changes,'matching_requires_regeneration':True}
OUTPUT.write_bytes(encoded)
REPORT.write_text(json.dumps(report,ensure_ascii=False,indent=2),encoding='utf-8')
print(OUTPUT, REPORT)

## 4. Suite
Régénérer les 2 000 couples avec le catalogue des 40 plantes, réévaluer les labels et réentraîner le Random Forest si applicable. Ne pas modifier les photos ni le dataset ResNet-18. Vérifier les enums autorisés et les unités dans GDAI-PLANT avant import.